In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from data.loader import load_tables, load_excel_files
from data.calculations import (
    build_product_classification,
    build_product_families,
    enrich_sales_with_classification,
    build_customer_activity,
    enrich_customers_with_activity,
    enrich_sales_with_customer_activity,
    filter_sales_by_bodega,
    prepare_sales
)

# -----------------------------
# Load raw data
# -----------------------------
access_file = "/Users/ricardolugo/Library/CloudStorage/OneDrive-Personal/LH/Reports/sales_lh.accdb"
tables = ["sales", "customers"]

data = load_tables(access_file, tables)

df_sales = data["sales"]
df_customers = data["customers"]
df_actividades, df_clasificacion, df_inventory = load_excel_files()

# -----------------------------
# Build reference tables
# -----------------------------
df_grupos = build_product_classification(df_clasificacion)
df_familias = build_product_families(df_clasificacion)
df_customer_activity = build_customer_activity(df_actividades)

# -----------------------------
# Enrich tables
# -----------------------------
df_sales_enriched = enrich_sales_with_classification(
    df_sales,
    df_grupos,
    df_familias
)

df_customers_enriched = enrich_customers_with_activity(
    df_customers,
    df_customer_activity
)

df_sales_final = enrich_sales_with_customer_activity(
    df_sales_enriched,
    df_customers_enriched
)

# -----------------------------
# Working filter: Cali
# -----------------------------

df_sales_clean = prepare_sales(df_sales_final)

df_cali = filter_sales_by_bodega(df_sales_clean, 50)
df_bogota = filter_sales_by_bodega(df_sales_clean, 1)

Loading sales...
sales loaded: (285619, 25)
Loading customers...
customers loaded: (36429, 16)
Actividades shape: (594, 6)
Clasificacion shape: (203, 4)
Inventario shape: (11169, 32)


/Users/ricardolugo/Documents/dev/LH/project_cali/data/calculations.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["fecha"] = pd.to_datetime(


In [2]:
# -----------------------------
# BLOQUE 1: TAMANO DEL NEGOCIO CALI
# -----------------------------

import pandas as pd

# Asegurar tipos

df_cali_analysis = df_cali.copy()

df_cali_analysis["fecha"] = pd.to_datetime(df_cali_analysis["fecha"], errors="coerce")
df_cali_analysis["valorbruto"] = pd.to_numeric(df_cali_analysis["valorbruto"], errors="coerce")
df_cali_analysis["cantidad"] = pd.to_numeric(df_cali_analysis["cantidad"], errors="coerce")
df_cali_analysis["precio"] = pd.to_numeric(df_cali_analysis["precio"], errors="coerce")
df_cali_analysis["costo"] = pd.to_numeric(df_cali_analysis["costo"], errors="coerce")

# Helpers
df_cali_analysis["year"] = df_cali_analysis["fecha"].dt.year
df_cali_analysis["year_month"] = df_cali_analysis["fecha"].dt.to_period("M")


In [3]:
df_cali_analysis["ventas_mm"] = df_cali_analysis["valorbruto"] / 1_000_000

# -----------------------------
# RUN RATE (ULTIMOS MESES)
# -----------------------------

today = pd.Timestamp("2026-04-01")

df_cali_analysis["fecha"] = pd.to_datetime(
    df_cali_analysis["fecha"],
    format="%d/%m/%y %H:%M:%S",
    errors="coerce"
)

#df_cali_analysis["ventas_mm"] = pd.to_numeric(df_cali_analysis["valorbruto"], errors="coerce")

# últimos 3 meses
df_last_3m = df_cali_analysis[
    df_cali_analysis["fecha"] >= today - pd.DateOffset(months=3)
]

# últimos 6 meses
df_last_6m = df_cali_analysis[
    df_cali_analysis["fecha"] >= today - pd.DateOffset(months=6)
]

print("Ventas últimos 3 meses:", f'{df_last_3m["ventas_mm"].sum():,.0f}')
print("Ventas últimos 6 meses:", f'{df_last_6m["ventas_mm"].sum():,.0f}')

df_cali_analysis

Ventas últimos 3 meses: 1,331
Ventas últimos 6 meses: 2,737


,nit,razonsocial,prefijo,numero,fecha,sucursal,idconvenio,ordencompra,sku,nombreproducto,...,descuento,vdescuento,costo,id,nombre_grupo,nombre_familia,actividad_economica,year,year_month,ventas_mm
249,890324697,INDUSTRIAS JILA LTDA,SC2,1004380,2024-02-09,0,,27502,6007-C-2Z-L038-C3FAG,RODAMIENTO RIGIDO DE BOLAS,...,0,0,14526.529674,255,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS,2024,2024-02,0.0204
250,890324697,INDUSTRIAS JILA LTDA,SC2,1004380,2024-02-09,0,,27502,K 21X25X13-A/0-7INA,CORONA DE AGUJAS,...,0,0,6996.122067,256,RODAMIENTOS DE AGUJAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS,2024,2024-02,0.0226
251,890324697,INDUSTRIAS JILA LTDA,SC2,1004381,2024-02-09,0,,27475,NK 20/20-XLINA,RODAMIENTO DE AGUJAS,...,0,0,21858.613169,257,RODAMIENTOS DE AGUJAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS,2024,2024-02,0.2632
252,890907841,ALMACEN RODAMIENTOS S.A.,SC2,1004382,2024-02-09,2,,2100,6203-2Z/GJNSKF,RODAMIENTO RIGIDO DE BOLAS,...,0,0,8812.945068,258,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,COMERCIO EN GENERAL,2024,2024-02,0.1000
253,890324697,INDUSTRIAS JILA LTDA,SC2,1004383,2024-02-09,0,,27505,51105NTN,RODAMIENTO AXIAL DE BOLAS,...,0,0,15609.121758,259,RODAMIENTO AXIAL DE BOLAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS,2024,2024-02,0.0223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285600,827000108,SOCIEDAD PRODUCTORA DE ENERGIA DE SAN ANDRES Y...,SZ2,22,2026-03-31,0,,19598,UCF 208-24SKF,SOPORTE Y RODAMIENTO,...,0,0,85742.646086,285712,NaN,CHUMACERAS,GENERACION DE ENERGIA,2026,2026-03,0.4320
285601,827000108,SOCIEDAD PRODUCTORA DE ENERGIA DE SAN ANDRES Y...,SZ2,23,2026-03-31,0,,19570,6303-2Z/GJNSKF,RODAMIENTO RIGIDO DE BOLAS,...,0,0,11405.069917,285713,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,GENERACION DE ENERGIA,2026,2026-03,0.2980
285609,830144577,ABC RODAMIENTOS Y SUMINISTROS INDUSTRIALES S.A.S.,VOC,2576,2026-03-31,0,,,NKX 30-Z-XLINA,RODAMIENTO COMBINADO,...,0,0,111131.673444,285721,RODAMIENTOS DE AGUJAS,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES,2026,2026-03,0.6156
285610,800078269,RODANDO S.A.S. BIC,VOC,2577,2026-03-31,0,,,1202 ETN9SKF,RODAMIENTO DE BOLAS A ROTULA,...,0,17340,81737.383837,285722,RODAMIENTO DE BOLAS A ROTULA,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES,2026,2026-03,0.3468


In [4]:
ventas_cliente = (
    df_last_6m.groupby("nit")["ventas_mm"]
    .sum()
    .sort_values(ascending=False)
)

top10_pct = ventas_cliente.head(10).sum() / ventas_cliente.sum()

print("Top 10 clientes %:", round(top10_pct, 2))

Top 10 clientes %: 0.34


In [5]:
df_last_6m.groupby("nombre_familia")["ventas_mm"] \
    .sum() \
    .sort_values(ascending=False) \
    .head(15)

nombre_familia
RODAMIENTOS                1996.756308
CHUMACERAS                  308.912045
TRANSMISION DE POTENCIA     190.423069
ACCESORIOS                  151.197850
RETENEDORES                  34.228171
LUBRICACION                  31.601879
SERVICIOS                    21.914941
OBSOLESCENCIAS                2.012800
Name: ventas_mm, dtype: float64

In [6]:
df_cali_analysis["year_month"] = df_cali_analysis["fecha"].dt.to_period("M")

pivot = pd.pivot_table(
    df_cali_analysis,
    index="nit",
    columns="year_month",
    values="ventas_mm",
    aggfunc="sum",
    fill_value=0
)

recent = pivot.iloc[:, -3:].sum(axis=1)
previous = pivot.iloc[:, -6:-3].sum(axis=1)

growth_df = pd.DataFrame({
    "ventas_prev_3m": previous,
    "ventas_recent_3m": recent
})

growth_df["growth_abs"] = growth_df["ventas_recent_3m"] - growth_df["ventas_prev_3m"]
growth_df["growth_pct"] = growth_df["growth_abs"] / growth_df["ventas_prev_3m"].replace(0, pd.NA)

customer_names = (
    df_cali_analysis[["nit", "razonsocial"]]
    .drop_duplicates(subset=["nit"])
)

growth_df = growth_df.reset_index().merge(customer_names, on="nit", how="left")

growth_df.sort_values("growth_abs").head(10)

,nit,ventas_prev_3m,ventas_recent_3m,growth_abs,growth_pct,razonsocial
915,900087414,21.125407,0.000000,-21.125407,-1.0,RIOPAILA CASTILLA S.A.
798,890300406,10.388900,0.000000,-10.388900,-1.0,CARTON DE COLOMBIA S.A.
643,805015031,8.427700,0.000000,-8.427700,-1.0,MYM BOBINADOS INDUSTRIALES SAS
807,890301886,7.897000,0.340700,-7.556300,-0.956857,FABRICA NACIONAL DE AUTOPARTES S.A. - FANALCA ...
883,891300238,6.923800,0.230400,-6.693400,-0.966723,INGENIO PROVIDENCIA S.A.
760,860002130,7.421800,1.410000,-6.011800,-0.810019,NESTLE DE COLOMBIA S.A.
776,860026759,11.814638,8.074555,-3.740083,-0.316563,CARTONES AMERICA S.A.
690,810006746,2.999000,0.000000,-2.999000,-1.0,TECNOBOBINADOS S.A.S.
806,890301884,2.965416,0.000000,-2.965416,-1.0,COLOMBINA S.A
811,890302594,2.893400,0.000000,-2.893400,-1.0,MAYAGUEZ S.A.


In [7]:
growth_df.sort_values("growth_abs").head(20)

,nit,ventas_prev_3m,ventas_recent_3m,growth_abs,growth_pct,razonsocial
915,900087414,21.125407,0.000000,-21.125407,-1.0,RIOPAILA CASTILLA S.A.
798,890300406,10.388900,0.000000,-10.388900,-1.0,CARTON DE COLOMBIA S.A.
643,805015031,8.427700,0.000000,-8.427700,-1.0,MYM BOBINADOS INDUSTRIALES SAS
807,890301886,7.897000,0.340700,-7.556300,-0.956857,FABRICA NACIONAL DE AUTOPARTES S.A. - FANALCA ...
883,891300238,6.923800,0.230400,-6.693400,-0.966723,INGENIO PROVIDENCIA S.A.
760,860002130,7.421800,1.410000,-6.011800,-0.810019,NESTLE DE COLOMBIA S.A.
776,860026759,11.814638,8.074555,-3.740083,-0.316563,CARTONES AMERICA S.A.
690,810006746,2.999000,0.000000,-2.999000,-1.0,TECNOBOBINADOS S.A.S.
806,890301884,2.965416,0.000000,-2.965416,-1.0,COLOMBINA S.A
811,890302594,2.893400,0.000000,-2.893400,-1.0,MAYAGUEZ S.A.


In [8]:
recent_clients = set(df_last_3m["nit"])
prev_clients = set(df_last_6m["nit"])

lost_clients = prev_clients - recent_clients

len(lost_clients)

187

In [9]:
df_last_3m.groupby("nombre_familia")["ventas_mm"] \
    .sum().sort_values(ascending=False).head(10)

nombre_familia
RODAMIENTOS                999.535760
CHUMACERAS                 151.109993
TRANSMISION DE POTENCIA     89.711938
ACCESORIOS                  36.202607
LUBRICACION                 20.602623
RETENEDORES                 16.728861
SERVICIOS                   16.318923
OBSOLESCENCIAS               0.694400
Name: ventas_mm, dtype: float64

In [10]:
lost_df = growth_df.sort_values("growth_abs").head(20)

lost_df[["nit", "razonsocial", "ventas_prev_3m", "ventas_recent_3m", "growth_abs"]]

,nit,razonsocial,ventas_prev_3m,ventas_recent_3m,growth_abs
915,900087414,RIOPAILA CASTILLA S.A.,21.125407,0.000000,-21.125407
798,890300406,CARTON DE COLOMBIA S.A.,10.388900,0.000000,-10.388900
643,805015031,MYM BOBINADOS INDUSTRIALES SAS,8.427700,0.000000,-8.427700
807,890301886,FABRICA NACIONAL DE AUTOPARTES S.A. - FANALCA ...,7.897000,0.340700,-7.556300
883,891300238,INGENIO PROVIDENCIA S.A.,6.923800,0.230400,-6.693400
760,860002130,NESTLE DE COLOMBIA S.A.,7.421800,1.410000,-6.011800
776,860026759,CARTONES AMERICA S.A.,11.814638,8.074555,-3.740083
690,810006746,TECNOBOBINADOS S.A.S.,2.999000,0.000000,-2.999000
806,890301884,COLOMBINA S.A,2.965416,0.000000,-2.965416
811,890302594,MAYAGUEZ S.A.,2.893400,0.000000,-2.893400


In [11]:
# clientes activos últimos 3 meses
active_clients = set(df_last_3m["nit"])

# clientes en riesgo (estaban en 6m pero no en 3m)
at_risk = set(df_last_6m["nit"]) - active_clients

# clientes totalmente perdidos (históricos vs 6m)
historical = set(df_cali_analysis["nit"])

lost = historical - set(df_last_6m["nit"])

print("Activos:", len(active_clients))
print("En riesgo:", len(at_risk))
print("Perdidos:", len(lost))

Activos: 395
En riesgo: 187
Perdidos: 1036


In [12]:
lost_clients_df = df_cali_analysis[
    df_cali_analysis["nit"].isin(lost)
]

lost_value = (
    lost_clients_df.groupby("nit")["ventas_mm"]
    .sum()
    .sort_values(ascending=False)
)

lost_value_df = lost_value.reset_index()

# traer razon social
customer_names = (
    df_cali_analysis[["nit", "razonsocial"]]
    .drop_duplicates(subset=["nit"])
)

lost_value_df = lost_value_df.merge(
    customer_names,
    on="nit",
    how="left"
)

# ordenar nuevamente
lost_value_df = lost_value_df.sort_values("ventas_mm", ascending=False)

lost_value_df.head(20)

,nit,ventas_mm,razonsocial
0,830511110,77.710885,SKF LATIN TRADE S.A.S.
1,16577584,65.701398,DELPIANY MACERO YOGER DANIEL
2,901210865,52.632450,AGROBANDAS & RODAMIENTOS SAS
3,805030163,35.419400,FERRE INDUSTRIAL LIMITADA
4,900950686,30.947254,OCCIDENTAL DE FINANCIACION S.A.S.
5,817000680,29.276736,FAMILIA SANCELA DEL PACIFICO S.A.S
6,860005264,22.557900,GRASCO LTDA
7,891304849,20.104300,INGENIERIA MAQUINARIA Y EQUIPOS DE COLOMBIA S....
8,830002366,20.027300,BIMBO DE COLOMBIA S.A.
9,890321924,19.955150,CINTAS ANDINAS DE COLOMBIA S.A.


In [13]:
df_cali_analysis["segmento_cliente"] = df_cali_analysis["nit"].apply(
    lambda x: "activo" if x in active_clients
    else "en_riesgo" if x in at_risk
    else "perdido"
)

df_cali_analysis.groupby("segmento_cliente")["ventas_mm"] \
    .sum()

segmento_cliente
activo       16327.306899
en_riesgo     1333.269961
perdido       1230.062297
Name: ventas_mm, dtype: float64

In [14]:
customer_vendor = (
    df_customers_enriched[["nit", "razonsocial", "vendedor"]]
    .drop_duplicates(subset=["nit"])
    .copy()
)

lost_value_df = lost_value_df.merge(
    customer_vendor[["nit", "vendedor"]],
    on="nit",
    how="left"
)

lost_value_df[["nit", "razonsocial", "ventas_mm", "vendedor"]].head(20)

,nit,razonsocial,ventas_mm,vendedor
0,830511110,SKF LATIN TRADE S.A.S.,77.710885,ALMACEN
1,16577584,DELPIANY MACERO YOGER DANIEL,65.701398,ALMACEN CALI -UNO-
2,901210865,AGROBANDAS & RODAMIENTOS SAS,52.632450,NUBIA ANDREA JIMENEZ
3,805030163,FERRE INDUSTRIAL LIMITADA,35.419400,NUBIA ANDREA JIMENEZ
4,900950686,OCCIDENTAL DE FINANCIACION S.A.S.,30.947254,YEISSON ANDRES RENTERIA MOSQUERA
5,817000680,FAMILIA SANCELA DEL PACIFICO S.A.S,29.276736,JAIRO DAVID VERA
6,860005264,GRASCO LTDA,22.557900,YEISSON ANDRES RENTERIA MOSQUERA
7,891304849,INGENIERIA MAQUINARIA Y EQUIPOS DE COLOMBIA S....,20.104300,ALMACEN CALI -UNO-
8,830002366,BIMBO DE COLOMBIA S.A.,20.027300,JEISMAN HOLGUIN
9,890321924,CINTAS ANDINAS DE COLOMBIA S.A.,19.955150,NUBIA ANDREA JIMENEZ


In [15]:
lost_by_vendor = (
    lost_value_df.groupby("vendedor", dropna=False)["ventas_mm"]
    .sum()
    .sort_values(ascending=False)
)

lost_by_vendor

vendedor
ALMACEN CALI -UNO-                  634.217905
ALMACEN                             186.466962
NUBIA ANDREA JIMENEZ                128.475930
JAIRO DAVID VERA                     67.030961
YEISSON ANDRES RENTERIA MOSQUERA     66.640552
JEISMAN HOLGUIN                      53.350163
NaN                                  51.420714
WHATSAPP CALI                        13.459600
JOSE TRINIDAD BELTRAN CARVAJAL        9.890900
EDWIN ARBELAEZ                        4.585000
JOHAN SEBASTIAN CORDOBA GAMBOA        3.892760
MARIA ELSA SIERRA FERNANDEZ           2.603240
NICOLAS LUGO GOMEZ                    2.108400
FABIO NELSON VALENCIA                 1.744500
WHATSAPP JOHN MORENO                  1.651650
CALL CENTER                           0.806900
WHATSAPP MARÍA                        0.677000
JHON ALEXANDER PINZON                 0.342000
TIENDA VIRTUAL                        0.323360
JOHN FREDY MORENO                     0.261100
FERNEY ORDOÑEZ                        0.087600
JUAN

In [16]:
lost_summary_vendor = (
    lost_value_df.groupby("vendedor", dropna=False)
    .agg(
        ventas_perdidas=("ventas_mm", "sum"),
        clientes_perdidos=("nit", "nunique")
    )
    .sort_values("ventas_perdidas", ascending=False)
)

lost_summary_vendor

,ventas_perdidas,clientes_perdidos
vendedor,,
ALMACEN CALI -UNO-,634.217905,694
ALMACEN,186.466962,106
NUBIA ANDREA JIMENEZ,128.475930,8
JAIRO DAVID VERA,67.030961,7
YEISSON ANDRES RENTERIA MOSQUERA,66.640552,9
JEISMAN HOLGUIN,53.350163,4
NaN,51.420714,173
WHATSAPP CALI,13.459600,8
JOSE TRINIDAD BELTRAN CARVAJAL,9.890900,1


In [17]:
top_clients_6m = (
    df_last_6m.groupby(["nit", "razonsocial"])["ventas_mm"]
    .sum()
    .reset_index()
    .sort_values("ventas_mm", ascending=False)
)

top_clients_6m.head(20)

,nit,razonsocial,ventas_mm
257,860026759,CARTONES AMERICA S.A.,212.977879
168,800201914,SANCHEZ Y ESCOBAR RODAMIENTOS S.A.S.,141.306030
413,900611187,SERMOTOR INGENIERIA S.A.S,113.867748
269,890300406,CARTON DE COLOMBIA S.A.,99.399968
247,860002130,NESTLE DE COLOMBIA S.A.,70.260580
215,815002042,PGI COLOMBIA LTDA.,67.887301
172,800236700,RODICLAR S.A.S.,66.262040
312,890327072,IME INGENIERIA DE MAQUINAS ELECTRICAS S.A.S,62.647436
329,891400378,PAPELES NACIONALES S.A.S,54.950720
167,800197463,POLLOS EL BUCANERO S.A.,49.452309


In [18]:
top_clients_3m = (
    df_last_3m.groupby(["nit", "razonsocial"])["ventas_mm"]
    .sum()
    .reset_index()
    .sort_values("ventas_mm", ascending=False)
)

top_clients_3m.head(20)

,nit,razonsocial,ventas_mm
181,860026759,CARTONES AMERICA S.A.,149.201559
187,890300406,CARTON DE COLOMBIA S.A.,92.686888
115,800201914,SANCHEZ Y ESCOBAR RODAMIENTOS S.A.S.,68.089050
283,900611187,SERMOTOR INGENIERIA S.A.S,52.463427
150,815002042,PGI COLOMBIA LTDA.,46.438222
216,890327072,IME INGENIERIA DE MAQUINAS ELECTRICAS S.A.S,35.243740
174,860002130,NESTLE DE COLOMBIA S.A.,30.350880
117,800236700,RODICLAR S.A.S.,29.006660
101,800078269,RODANDO S.A.S. BIC,28.971260
238,900087414,RIOPAILA CASTILLA S.A.,27.901757


In [19]:
top_6m_set = set(top_clients_6m.head(20)["nit"])
top_3m_set = set(top_clients_3m.head(20)["nit"])

lost_top_clients = top_6m_set - top_3m_set
new_top_clients = top_3m_set - top_6m_set

print("Top clientes que salieron del top 20:", len(lost_top_clients))
print("Nuevos en el top 20:", len(new_top_clients))

Top clientes que salieron del top 20: 5
Nuevos en el top 20: 5


In [20]:
top_clients_6m[top_clients_6m["nit"].isin(lost_top_clients)]

,nit,razonsocial,ventas_mm
372,900292211,REFINADORA NACIONAL DE ACEITES Y GRASAS S.A.S,44.726539
283,890302594,MAYAGUEZ S.A.,40.222285
210,805031716,CALI RODAMIENTOS S.A.S.,35.130460
187,805007524,COMERCIALIZADORA Y SERVICIOS PROFESIONALES S.A...,33.279540
423,900687323,ALMACEN RODAFER RF S.A.S.,32.056740
371,900292211,REFINADORA NACIONAL DE ACEITES Y GRASAS S.A.,0.594800


In [21]:
at_risk


{'1024569176',
 '1030521848',
 '10482610',
 '1085324557',
 '1096241082',
 '1109841180',
 '1112879822',
 '1114309267',
 '1115072866',
 '1118257206',
 '1118298006',
 '1130594819',
 '1151953675',
 '1151956511',
 '11936665',
 '14467134',
 '14624657',
 '14837593',
 '14944000',
 '15905581',
 '16258150',
 '16269571',
 '16289474',
 '16613411',
 '16661992',
 '16680945',
 '16690775',
 '16696293',
 '16710674',
 '16718692',
 '16759596',
 '16797355',
 '16831135',
 '16933714',
 '31228707',
 '31914280',
 '38437205',
 '4137377',
 '444444081',
 '6106571',
 '6219612',
 '66834437',
 '66853940',
 '67015316',
 '700378133',
 '70905784',
 '75064885',
 '7529104',
 '800013349',
 '800125352',
 '800169567',
 '800178801',
 '800208128',
 '800210144',
 '800253727',
 '802021585',
 '80240519',
 '80413960',
 '805016291',
 '805027036',
 '805027332',
 '805031667',
 '811035407',
 '817000790',
 '821001526',
 '830007248',
 '830046982',
 '830069897',
 '830070147',
 '830503491',
 '860002518',
 '860002523',
 '860003375',
 '86

In [22]:
# -----------------------------
# CLIENTES EN RIESGO
# estaban en últimos 6 meses, pero no compraron en últimos 3 meses
# -----------------------------

# 1. Base de clientes en riesgo
at_risk_df = df_cali_analysis[df_cali_analysis["nit"].isin(at_risk)].copy()

# 2. Ventas en últimos 6 meses por cliente
at_risk_sales_6m = (
    at_risk_df.groupby("nit", as_index=False)["ventas_mm"]
    .sum()
    .rename(columns={"ventas_mm": "ventas_ult_6m_mm"})
)

# 3. Razón social por cliente
customer_names = (
    df_cali_analysis[["nit", "razonsocial"]]
    .drop_duplicates(subset=["nit"])
    .copy()
)

# 4. Vendedor por cliente
# usando el vendedor del maestro de clientes enriquecido
customer_vendor = (
    df_customers_enriched[["nit", "vendedor"]]
    .copy()
)

customer_vendor["nit"] = customer_vendor["nit"].astype(str).str.strip()

customer_vendor = (
    customer_vendor.drop_duplicates(subset=["nit"])
)

# 5. Armar tabla final
at_risk_table = (
    at_risk_sales_6m
    .merge(customer_names, on="nit", how="left")
    .merge(customer_vendor, on="nit", how="left")
)

# 6. Confirmar que no compraron en últimos 3 meses
at_risk_table["compra_ult_3m"] = "NO"

# 7. Ordenar por valor
at_risk_table = at_risk_table.sort_values("ventas_ult_6m_mm", ascending=False)

# 8. Resultado
at_risk_table[["nit", "razonsocial", "ventas_ult_6m_mm", "compra_ult_3m", "vendedor"]].head(30)

,nit,razonsocial,ventas_ult_6m_mm,compra_ult_3m,vendedor
76,890210569,CENTRAL DE BOBINADOS S.A.,164.240540,NO,JAIRO DAVID VERA
90,891300382,HARINERA DEL VALLE S.A.,158.580846,NO,JAIRO DAVID VERA
79,890301463,LAFRANCOL S.A.S.,81.721478,NO,JAIRO DAVID VERA
53,800210144,INGENIO MARIA LUISA S.A.,66.625433,NO,JAIRO DAVID VERA
34,31228707,ARGOTE ARGOTE AURELIA,62.901110,NO,ALMACEN CALI -UNO-
96,900124999,RODAMIENTOS CJR S.A.S SUCESORES,60.884140,NO,ALMACEN CALI -UNO-
126,900783257,INAGROMIN S.A.S,59.392489,NO,JAIRO DAVID VERA
104,900315287,INGENIO DEL OCCIDENTE S.A.S.,50.852100,NO,JAIRO DAVID VERA
60,805027332,PRODUCTORA DE CONFITES Y CHICLET´S MAC DULCES...,46.505706,NO,NUBIA ANDREA JIMENEZ
78,890300768,LA BALINERA S.A EN REORGANIZACION,25.941980,NO,ALMACEN


In [23]:
at_risk_total_mm = at_risk_table["ventas_ult_6m_mm"].sum()
at_risk_clientes = at_risk_table["nit"].nunique()

print("Clientes en riesgo:", at_risk_clientes)
print("Ventas en riesgo últimos 6 meses (MM):", round(at_risk_total_mm, 1))

Clientes en riesgo: 187
Ventas en riesgo últimos 6 meses (MM): 1333.3


In [24]:
at_risk_by_vendor = (
    at_risk_table.groupby("vendedor", dropna=False)
    .agg(
        clientes_en_riesgo=("nit", "nunique"),
        ventas_en_riesgo_mm=("ventas_ult_6m_mm", "sum")
    )
    .sort_values("ventas_en_riesgo_mm", ascending=False)
)

at_risk_by_vendor

,clientes_en_riesgo,ventas_en_riesgo_mm
vendedor,,
JAIRO DAVID VERA,7,588.695586
ALMACEN CALI -UNO-,93,424.050778
NUBIA ANDREA JIMENEZ,7,98.186803
ALMACEN,23,97.543950
NaN,52,78.804463
JEISMAN HOLGUIN,2,29.546000
MAURICIO ALBERTO SUAREZ,1,14.123100
TIENDA VIRTUAL,1,2.319280
WHATSAPP MARÍA,1,0.000000


In [25]:
top20_at_risk = at_risk_table.head(20)

top20_at_risk[[
    "nit",
    "razonsocial",
    "ventas_ult_6m_mm",
    "vendedor"
]]

,nit,razonsocial,ventas_ult_6m_mm,vendedor
76,890210569,CENTRAL DE BOBINADOS S.A.,164.240540,JAIRO DAVID VERA
90,891300382,HARINERA DEL VALLE S.A.,158.580846,JAIRO DAVID VERA
79,890301463,LAFRANCOL S.A.S.,81.721478,JAIRO DAVID VERA
53,800210144,INGENIO MARIA LUISA S.A.,66.625433,JAIRO DAVID VERA
34,31228707,ARGOTE ARGOTE AURELIA,62.901110,ALMACEN CALI -UNO-
96,900124999,RODAMIENTOS CJR S.A.S SUCESORES,60.884140,ALMACEN CALI -UNO-
126,900783257,INAGROMIN S.A.S,59.392489,JAIRO DAVID VERA
104,900315287,INGENIO DEL OCCIDENTE S.A.S.,50.852100,JAIRO DAVID VERA
60,805027332,PRODUCTORA DE CONFITES Y CHICLET´S MAC DULCES...,46.505706,NUBIA ANDREA JIMENEZ
78,890300768,LA BALINERA S.A EN REORGANIZACION,25.941980,ALMACEN


In [26]:
at_risk_detail = (
    df_cali_analysis[df_cali_analysis["nit"].isin(at_risk)]
    .groupby(["nit", "razonsocial", "nombre_grupo"])["ventas_mm"]
    .sum()
    .reset_index()
)

# top producto por cliente
top_product_per_client = (
    at_risk_detail.sort_values(["nit", "ventas_mm"], ascending=[True, False])
    .drop_duplicates(subset=["nit"])
)

top20_at_risk = top20_at_risk.merge(
    top_product_per_client[["nit", "nombre_grupo"]],
    on="nit",
    how="left"
)

top20_at_risk

,nit,ventas_ult_6m_mm,razonsocial,vendedor,compra_ult_3m,nombre_grupo
0,890210569,164.240540,CENTRAL DE BOBINADOS S.A.,JAIRO DAVID VERA,NO,RODAMIENTOS RIGIDOS DE BOLAS
1,891300382,158.580846,HARINERA DEL VALLE S.A.,JAIRO DAVID VERA,NO,RODAMIENTOS DE RODILLOS A ROTULA
2,890301463,81.721478,LAFRANCOL S.A.S.,JAIRO DAVID VERA,NO,RODAMIENTOS DE AGUJAS
3,800210144,66.625433,INGENIO MARIA LUISA S.A.,JAIRO DAVID VERA,NO,RODAMIENTOS RIGIDOS DE BOLAS
4,31228707,62.901110,ARGOTE ARGOTE AURELIA,ALMACEN CALI -UNO-,NO,RODAMIENTOS RIGIDOS DE BOLAS
5,900124999,60.884140,RODAMIENTOS CJR S.A.S SUCESORES,ALMACEN CALI -UNO-,NO,RODAMIENTOS RIGIDOS DE BOLAS
6,900783257,59.392489,INAGROMIN S.A.S,JAIRO DAVID VERA,NO,RODAMIENTOS DE RODILLOS A ROTULA
7,900315287,50.852100,INGENIO DEL OCCIDENTE S.A.S.,JAIRO DAVID VERA,NO,RODAMIENTOS DE RODILLOS A ROTULA
8,805027332,46.505706,PRODUCTORA DE CONFITES Y CHICLET´S MAC DULCES...,NUBIA ANDREA JIMENEZ,NO,RODAMIENTOS RIGIDOS DE BOLAS
9,890300768,25.941980,LA BALINERA S.A EN REORGANIZACION,ALMACEN,NO,GUIAS Y RIELES LINEALES


In [27]:
at_risk_table.to_excel("at_risk_clients.xlsx", index=False)

In [28]:
almacen_customers = (
    df_customers_enriched[
        df_customers_enriched["vendedor"].astype(str).str.contains("ALMACEN CALI -UNO-", case=False, na=False)
    ][["nit", "razonsocial", "vendedor", "actividad_economica"]]
    .drop_duplicates(subset=["nit"])
    .copy()
)

almacen_customers.head()

,nit,razonsocial,vendedor,actividad_economica
14,94525759,TORRES VASQUEZ RICARDO,ALMACEN CALI -UNO-,COMERCIO ALIMENTOS Y BEBIDAS
16,10125826,GONZALEZ TAPASCO JUAN CARLOS,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES
23,900075205,SEGURIDAD ALIMENTARIA DE OCCIDENTE SAS,ALMACEN CALI -UNO-,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS
27,901398924,DAPA SOLUCIONES DE INGENIERÍA S.A.S.,ALMACEN CALI -UNO-,TALLERES
31,830055392,LAMAOS S.A.,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES


In [29]:
cutoff = df_cali_analysis["fecha"].max()

start_12m = cutoff - pd.DateOffset(months=12)
start_6m = cutoff - pd.DateOffset(months=6)
start_3m = cutoff - pd.DateOffset(months=3)

print("Cutoff:", cutoff)
print("12m start:", start_12m)
print("6m start:", start_6m)
print("3m start:", start_3m)

Cutoff: 2026-12-03 00:00:00
12m start: 2025-12-03 00:00:00
6m start: 2026-06-03 00:00:00
3m start: 2026-09-03 00:00:00


In [30]:
sales_12m = (
    df_cali_analysis[df_cali_analysis["fecha"] >= start_12m]
    .groupby("nit", as_index=False)["ventas_mm"]
    .sum()
    .rename(columns={"ventas_mm": "ventas_12m_mm"})
)

sales_6m = (
    df_cali_analysis[df_cali_analysis["fecha"] >= start_6m]
    .groupby("nit", as_index=False)["ventas_mm"]
    .sum()
    .rename(columns={"ventas_mm": "ventas_6m_mm"})
)

sales_3m = (
    df_cali_analysis[df_cali_analysis["fecha"] >= start_3m]
    .groupby("nit", as_index=False)["ventas_mm"]
    .sum()
    .rename(columns={"ventas_mm": "ventas_3m_mm"})
)

In [31]:
almacen_customer_timeline = (
    almacen_customers
    .merge(sales_12m, on="nit", how="left")
    .merge(sales_6m, on="nit", how="left")
    .merge(sales_3m, on="nit", how="left")
    .fillna({
        "ventas_12m_mm": 0,
        "ventas_6m_mm": 0,
        "ventas_3m_mm": 0
    })
)

almacen_customer_timeline.head(20)

,nit,razonsocial,vendedor,actividad_economica,ventas_12m_mm,ventas_6m_mm,ventas_3m_mm
0,94525759,TORRES VASQUEZ RICARDO,ALMACEN CALI -UNO-,COMERCIO ALIMENTOS Y BEBIDAS,0.000000,0.00000,0.00000
1,10125826,GONZALEZ TAPASCO JUAN CARLOS,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,0.000000,0.00000,0.00000
2,900075205,SEGURIDAD ALIMENTARIA DE OCCIDENTE SAS,ALMACEN CALI -UNO-,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,0.000000,0.00000,0.00000
3,901398924,DAPA SOLUCIONES DE INGENIERÍA S.A.S.,ALMACEN CALI -UNO-,TALLERES,0.000000,0.00000,0.00000
4,830055392,LAMAOS S.A.,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,0.000000,0.00000,0.00000
5,800118173,SERVIRODAMIENTOS LTDA - CALI,ALMACEN CALI -UNO-,COMERCIO Y MANTENIMIENTO DE AUTOMOTORES,18.566675,3.74826,2.51596
6,14441161,SUAREZ ROLDAN MARCO FIDEL,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,0.000000,0.00000,0.00000
7,901460869,LA MESETA LOGISTICS S.A.S.,ALMACEN CALI -UNO-,TRANSPORTE TERRESTRE,0.000000,0.00000,0.00000
8,901475898,HIPER SUMINISTROS E IMPORTACIONES S.A.S.,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,0.000000,0.00000,0.00000
9,900071983,CAODEL S.A.S.,ALMACEN CALI -UNO-,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,0.452400,0.00000,0.00000


In [32]:
almacen_customer_timeline = almacen_customer_timeline.sort_values(
    ["ventas_12m_mm", "ventas_6m_mm"],
    ascending=[False, False]
)

almacen_customer_timeline.head(20)

,nit,razonsocial,vendedor,actividad_economica,ventas_12m_mm,ventas_6m_mm,ventas_3m_mm
190,800201914,SANCHEZ Y ESCOBAR RODAMIENTOS S.A.S.,ALMACEN CALI -UNO-,COMERCIO EN GENERAL,84.375650,12.73730,8.23280
2972,800236700,RODICLAR S.A.S.,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,38.490360,4.27870,3.79100
158,890307885,PLASTICOS ESPECIALES S.A.S,ALMACEN CALI -UNO-,PRODUCTOS PLASTICOS,26.442812,10.79080,10.79080
240,805007524,COMERCIALIZADORA Y SERVICIOS PROFESIONALES S.A...,ALMACEN CALI -UNO-,COMERCIO EN GENERAL,25.261940,1.95530,0.37810
70,805031716,CALI RODAMIENTOS S.A.S.,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,23.726680,0.57330,0.01100
2133,900687323,ALMACEN RODAFER RF S.A.S,ALMACEN CALI -UNO-,COMERCIO Y MANTENIMIENTO DE AUTOMOTORES,22.366900,0.52720,0.52720
2967,890907841,ALMACEN RODAMIENTOS S.A.,ALMACEN CALI -UNO-,COMERCIO EN GENERAL,21.766480,0.00000,0.00000
370,890311323,BENAR & CIA S.A.S.,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,21.160400,0.00000,0.00000
480,901053016,LATINRODAMIENTOS S.A.S.,ALMACEN CALI -UNO-,COMERCIO EN GENERAL,19.937360,4.39110,1.31020
5,800118173,SERVIRODAMIENTOS LTDA - CALI,ALMACEN CALI -UNO-,COMERCIO Y MANTENIMIENTO DE AUTOMOTORES,18.566675,3.74826,2.51596


In [33]:
exclude_actividades = [
    "COMERCIO EN GENERAL",
    "COMERCIO DE RODAMIENTOS Y AFINES",
    "COMERCIO Y MANTENIMIENTO DE AUTOMOTORES"
]

almacen_customer_timeline_clean = almacen_customer_timeline[
    ~almacen_customer_timeline["actividad_economica"].isin(exclude_actividades)
].copy()

In [34]:
almacen_customer_timeline_clean.tail(100)

,nit,razonsocial,vendedor,actividad_economica,ventas_12m_mm,ventas_6m_mm,ventas_3m_mm
2860,805017238,RAPI EXPRESS COMERCIALIZADORA,ALMACEN CALI -UNO-,OTROS TIPOS DE MANUFACTURA,0.0,0.0,0.0
2862,805028511,CORPORACION IPS OCCIDENTE,ALMACEN CALI -UNO-,EDUCACION,0.0,0.0,0.0
2863,805029674,HYSA LTDA,ALMACEN CALI -UNO-,TALLERES,0.0,0.0,0.0
2864,805029906,FLOW SYSTEM LTDA,ALMACEN CALI -UNO-,TALLERES,0.0,0.0,0.0
2865,805030954,INGRAB S.A,ALMACEN CALI -UNO-,PRODUCTOS TEXTILES,0.0,0.0,0.0
...,...,...,...,...,...,...,...
3036,800159376,FABRIFOLDER S.A.S.,ALMACEN CALI -UNO-,FABRICACION PRODUCTOS CARTON Y PAPEL,0.0,0.0,0.0
3037,14877449,GOMEZ JESUS MARIA,ALMACEN CALI -UNO-,AGRICULTURA,0.0,0.0,0.0
3038,805017220,INGENIERIA ELECTROMEDICA Y ANBIENTAL LTDA,ALMACEN CALI -UNO-,CONSTRUCTORAS,0.0,0.0,0.0
3040,900496821,GELECTROGENO S.A.S,ALMACEN CALI -UNO-,CONSTRUCTORAS,0.0,0.0,0.0


In [35]:
len(almacen_customer_timeline_clean)

1581

In [36]:
almacen_customer_timeline_clean.to_excel("almacen_customer_timeline_clean.xlsx", index=False)

In [37]:
almacen_customer_timeline_clean.head(100)

,nit,razonsocial,vendedor,actividad_economica,ventas_12m_mm,ventas_6m_mm,ventas_3m_mm
158,890307885,PLASTICOS ESPECIALES S.A.S,ALMACEN CALI -UNO-,PRODUCTOS PLASTICOS,26.442812,10.7908,10.7908
319,890319047,CARVAJAL EMPAQUES S.A.,ALMACEN CALI -UNO-,PRODUCTOS PLASTICOS,11.557180,0.2187,0.1838
3021,815003680,PRONALCUR S.A.S.,ALMACEN CALI -UNO-,"CURTIDO DE CUEROS, ARTICULOS DE CUERO",4.384900,0.7811,0.0356
1880,836000742,TRITURADOS Y CONCRETOS LTDA,ALMACEN CALI -UNO-,CONSTRUCTORAS,3.788800,0.2467,0.8868
115,900131688,ALINEAR MANTENIMIENTO S.A.S,ALMACEN CALI -UNO-,TALLERES,3.734100,0.0000,0.0000
...,...,...,...,...,...,...,...
2558,900694610,INGENIERIA Y MECANIZADOS DE COLOMBIA S.A.S,ALMACEN CALI -UNO-,TALLERES,0.065200,0.0000,0.0000
1567,900012691,CALDERAS HB S.A.S.,ALMACEN CALI -UNO-,TALLERES,0.065100,0.0000,0.0000
2471,900679672,CONSTRUCCIONES INDUSTRIALES ESPECIALIZADAS COI...,ALMACEN CALI -UNO-,CONSTRUCTORAS,0.061200,0.0612,0.0000
831,900012760,PROTENAL CALI SAS,ALMACEN CALI -UNO-,INDUSTRIAS METALURGICAS,0.057200,0.0000,0.0000


In [38]:
df_sales_final

,nit,razonsocial,prefijo,numero,fecha,sucursal,idconvenio,ordencompra,idproducto,nombreproducto,...,idfam1,neto,valorbruto,descuento,vdescuento,costo,id,nombre_grupo,nombre_familia,actividad_economica
0,830031093,FERREAMERICA S.A.S. REPRESENTACIONES,DE,5175,09/02/24 00:00:00,0,,,6208-2Z/C3GJNSKF,RODAMIENTO RIGIDO DE BOLAS,...,1000,-120666,-101400,0,0,37629.2078405458,1,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES
1,830095991,PARTES Y RODAMIENTOS LTDA,DE,5176,09/02/24 00:00:00,0,,,RB 1/4KMK,ESFERA DE ACERO,...,3000,-13328,-11200,0,0,12.5374319650662,2,NaN,ACCESORIOS,COMERCIO Y MANTENIMIENTO DE AUTOMOTORES
2,900104550,CAFE DEVOTION LTDA,DV,14990,09/02/24 00:00:00,0,,,LGHP 2/1SKF,GRASA PARA ALTA TEMPERATURA,...,6000,-228361,-191900,0,0,119356.879156687,3,NaN,LUBRICACION,COMERCIO ALIMENTOS Y BEBIDAS
3,830144577,ABC RODAMIENTOS Y SUMINISTROS INDUSTRIALES S.A.S.,FC2,1008169,09/02/24 00:00:00,0,,4925,6313-2RSR-C3FAG,RODAMIENTO RIGIDO DE BOLAS,...,1000,328440,276000,0,0,181394.644010172,4,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES
4,830144577,ABC RODAMIENTOS Y SUMINISTROS INDUSTRIALES S.A.S.,FC2,1008169,09/02/24 00:00:00,0,,4925,6314-2RSR-C3FAG,RODAMIENTO RIGIDO DE BOLAS,...,1000,391510,329000,0,0,220183.253830463,5,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285614,800078269,RODANDO S.A.S. BIC,VOC,2581,03/31/26 00:00:00,0,,,PHG C118SKF,CORREA EN V,...,5000,112782.25,669000,0,16725,50401.1401188285,285726,NaN,TRANSMISION DE POTENCIA,COMERCIO DE RODAMIENTOS Y AFINES
285615,800078935,IMPORTADORA INDUSTRIAL COLOMBIANA S.A.S IMPORINCO,VOC,2582,03/31/26 00:00:00,0,,,KM 4SKF,TUERCA DE FIJACION,...,3000,20634.6,20400,0,3060,16155.5295619127,285727,NaN,ACCESORIOS,COMERCIO DE RODAMIENTOS Y AFINES
285616,901331547,IMPORTACIONES SUMEXCO S.A.S.,VOC,2583,03/31/26 00:00:00,0,,,UK 25-ZZCTS,RODAMIENTO DE MARCHA LIBRE,...,1000,131614,221200,0,0,52781.0362226475,285728,RODAMIENTOS DE TRINQUETE,RODAMIENTOS,COMERCIO EN GENERAL
285617,800078935,IMPORTADORA INDUSTRIAL COLOMBIANA S.A.S IMPORINCO,VOC,2584,03/31/26 00:00:00,0,,,6201-2ZKMK,RODAMIENTO RIGIDO DE BOLAS,...,1000,1190,500000,0,0,496.749228205917,285729,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES


In [39]:
df_cali

,nit,razonsocial,prefijo,numero,fecha,sucursal,idconvenio,ordencompra,sku,nombreproducto,...,idfam1,neto,valorbruto,descuento,vdescuento,costo,id,nombre_grupo,nombre_familia,actividad_economica
249,890324697,INDUSTRIAS JILA LTDA,SC2,1004380,2024-02-09,0,,27502,6007-C-2Z-L038-C3FAG,RODAMIENTO RIGIDO DE BOLAS,...,1000,24276,20400,0,0,14526.5296737608,255,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS
250,890324697,INDUSTRIAS JILA LTDA,SC2,1004380,2024-02-09,0,,27502,K 21X25X13-A/0-7INA,CORONA DE AGUJAS,...,1000,26894,22600,0,0,6996.12206666667,256,RODAMIENTOS DE AGUJAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS
251,890324697,INDUSTRIAS JILA LTDA,SC2,1004381,2024-02-09,0,,27475,NK 20/20-XLINA,RODAMIENTO DE AGUJAS,...,1000,313208,263200,0,0,21858.6131686646,257,RODAMIENTOS DE AGUJAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS
252,890907841,ALMACEN RODAMIENTOS S.A.,SC2,1004382,2024-02-09,2,,2100,6203-2Z/GJNSKF,RODAMIENTO RIGIDO DE BOLAS,...,1000,119000,100000,0,0,8812.945067963101,258,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,COMERCIO EN GENERAL
253,890324697,INDUSTRIAS JILA LTDA,SC2,1004383,2024-02-09,0,,27505,51105NTN,RODAMIENTO AXIAL DE BOLAS,...,1000,26537,22300,0,0,15609.121757973,259,RODAMIENTO AXIAL DE BOLAS,RODAMIENTOS,EXTRACCION DE MINERALES METALIFEROS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285600,827000108,SOCIEDAD PRODUCTORA DE ENERGIA DE SAN ANDRES Y...,SZ2,22,2026-03-31,0,,19598,UCF 208-24SKF,SOPORTE Y RODAMIENTO,...,2000,432000,432000,0,0,85742.6460864649,285712,NaN,CHUMACERAS,GENERACION DE ENERGIA
285601,827000108,SOCIEDAD PRODUCTORA DE ENERGIA DE SAN ANDRES Y...,SZ2,23,2026-03-31,0,,19570,6303-2Z/GJNSKF,RODAMIENTO RIGIDO DE BOLAS,...,1000,298000,298000,0,0,11405.0699168081,285713,RODAMIENTOS RIGIDOS DE BOLAS,RODAMIENTOS,GENERACION DE ENERGIA
285609,830144577,ABC RODAMIENTOS Y SUMINISTROS INDUSTRIALES S.A.S.,VOC,2576,2026-03-31,0,,,NKX 30-Z-XLINA,RODAMIENTO COMBINADO,...,1000,183141,615600,0,0,111131.673443849,285721,RODAMIENTOS DE AGUJAS,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES
285610,800078269,RODANDO S.A.S. BIC,VOC,2577,2026-03-31,0,,,1202 ETN9SKF,RODAMIENTO DE BOLAS A ROTULA,...,1000,116929.4,346800,0,17340,81737.3838367243,285722,RODAMIENTO DE BOLAS A ROTULA,RODAMIENTOS,COMERCIO DE RODAMIENTOS Y AFINES


In [51]:
import pandas as pd
import numpy as np

# =========================================================
# BASE CLIENTES CALI
# usando valorbruto * cantidad (NO neto)
# =========================================================

df = df_cali.copy()

# ---------------------------------------------------------
# tipos
# ---------------------------------------------------------
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

numeric_cols = [
    "valorbruto",
    "cantidad",
    "costo",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# si cantidad viene vacía -> asumir 1
df.loc[df["cantidad"] <= 0, "cantidad"] = 1

# ---------------------------------------------------------
# ventas reales
# valorbruto = valor unitario
# venta total línea = valorbruto * cantidad
# costo total línea = costo * cantidad
# ---------------------------------------------------------

df["venta_linea"] = df["valorbruto"] * df["cantidad"]
df["costo_linea"] = df["costo"]
df["gross_profit"] = df["venta_linea"] - df["costo_linea"]

# ---------------------------------------------------------
# fecha referencia
# ---------------------------------------------------------

max_date = df["fecha"].max()

start_12m = max_date - pd.DateOffset(months=12)
start_6m = max_date - pd.DateOffset(months=6)
start_prev_12m = start_12m - pd.DateOffset(months=12)

df_12m = df[df["fecha"] >= start_12m].copy()
df_6m = df[df["fecha"] >= start_6m].copy()
df_prev_12m = df[
    (df["fecha"] >= start_prev_12m) &
    (df["fecha"] < start_12m)
].copy()

# =========================================================
# BASE HISTÓRICA
# =========================================================

base = (
    df.groupby(
        ["nit", "razonsocial"],
        as_index=False
    )
    .agg(
        primera_compra=("fecha", "min"),
        ultima_compra=("fecha", "max"),

        ventas_historicas=("venta_linea", "sum"),
        utilidad_historica=("gross_profit", "sum"),

        facturas_historicas=("numero", "nunique"),
        lineas_historicas=("sku", "count"),
        skus_historicos=("sku", "nunique"),

        actividad_economica=("actividad_economica", "last"),
    )
)

# =========================================================
# ÚLTIMOS 12 MESES
# =========================================================

agg_12m = (
    df_12m.groupby(
        ["nit", "razonsocial"],
        as_index=False
    )
    .agg(
        ventas_12m=("venta_linea", "sum"),
        utilidad_12m=("gross_profit", "sum"),

        facturas_12m=("numero", "nunique"),
        lineas_12m=("sku", "count"),
        skus_12m=("sku", "nunique"),

        ultima_compra_12m=("fecha", "max"),
    )
)

# =========================================================
# ÚLTIMOS 6 MESES
# =========================================================

agg_6m = (
    df_6m.groupby(
        ["nit", "razonsocial"],
        as_index=False
    )
    .agg(
        ventas_6m=("venta_linea", "sum"),
        utilidad_6m=("gross_profit", "sum"),

        facturas_6m=("numero", "nunique"),
        skus_6m=("sku", "nunique"),
    )
)

# =========================================================
# 12 MESES ANTERIORES
# para tendencia
# =========================================================

agg_prev_12m = (
    df_prev_12m.groupby(
        ["nit", "razonsocial"],
        as_index=False
    )
    .agg(
        ventas_prev_12m=("venta_linea", "sum"),
        utilidad_prev_12m=("gross_profit", "sum"),
        facturas_prev_12m=("numero", "nunique"),
    )
)

# =========================================================
# MERGE
# =========================================================

resumen_clientes = (
    base
    .merge(
        agg_12m,
        on=["nit", "razonsocial"],
        how="left"
    )
    .merge(
        agg_6m,
        on=["nit", "razonsocial"],
        how="left"
    )
    .merge(
        agg_prev_12m,
        on=["nit", "razonsocial"],
        how="left"
    )
)

# ---------------------------------------------------------
# fillna
# ---------------------------------------------------------

cols_fill_zero = [
    "ventas_12m",
    "utilidad_12m",
    "facturas_12m",
    "lineas_12m",
    "skus_12m",

    "ventas_6m",
    "utilidad_6m",
    "facturas_6m",
    "skus_6m",

    "ventas_prev_12m",
    "utilidad_prev_12m",
    "facturas_prev_12m",
]

for col in cols_fill_zero:
    resumen_clientes[col] = resumen_clientes[col].fillna(0)

# =========================================================
# MÉTRICAS GERENCIALES
# =========================================================

resumen_clientes["dias_sin_comprar"] = (
    max_date - resumen_clientes["ultima_compra"]
).dt.days

resumen_clientes["ticket_promedio_12m"] = np.where(
    resumen_clientes["facturas_12m"] > 0,
    resumen_clientes["ventas_12m"] / resumen_clientes["facturas_12m"],
    0
)

resumen_clientes["margen_bruto_pct_12m"] = np.where(
    resumen_clientes["ventas_12m"] > 0,
    resumen_clientes["utilidad_12m"] / resumen_clientes["ventas_12m"],
    0
)

resumen_clientes["variacion_ventas_12m_vs_prev"] = np.where(
    resumen_clientes["ventas_prev_12m"] > 0,
    (
        resumen_clientes["ventas_12m"] /
        resumen_clientes["ventas_prev_12m"]
    ) - 1,
    np.nan
)

resumen_clientes["frecuencia_facturas_mes_12m"] = (
    resumen_clientes["facturas_12m"] / 12
)

# =========================================================
# ESTADO COMERCIAL
# =========================================================

conditions = [
    (
        (resumen_clientes["ventas_12m"] > 0) &
        (resumen_clientes["dias_sin_comprar"] <= 60)
    ),
    (
        (resumen_clientes["ventas_12m"] > 0) &
        (resumen_clientes["dias_sin_comprar"].between(61, 120))
    ),
    (
        (resumen_clientes["ventas_12m"] > 0) &
        (resumen_clientes["dias_sin_comprar"] > 120)
    ),
    (
        (resumen_clientes["ventas_12m"] == 0) &
        (resumen_clientes["ventas_historicas"] > 0)
    ),
]

choices = [
    "Activa",
    "En Riesgo",
    "Dormida",
    "Sin compra reciente",
]

resumen_clientes["estado_comercial"] = np.select(
    conditions,
    choices,
    default="Nueva / sin historia reciente"
)

# =========================================================
# RANKING
# =========================================================

resumen_clientes = (
    resumen_clientes
    .sort_values(
        ["ventas_12m", "utilidad_12m"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

resumen_clientes["rank_ventas_12m"] = (
    resumen_clientes.index + 1
)

# =========================================================
# TOP 30 CUENTAS CALI
# =========================================================

# mostrar razón social primero y ocultar nit si no lo necesitas

cols_view = [
    "rank_ventas_12m",
    "razonsocial",
    # "nit",  # comentar o eliminar si no quieres mostrarlo

    "actividad_economica",

    "ventas_12m",
    "ventas_6m",
    "utilidad_12m",
    "margen_bruto_pct_12m",

    "facturas_12m",
    "skus_12m",
    "ticket_promedio_12m",

    "ultima_compra",
    "dias_sin_comprar",

    "variacion_ventas_12m_vs_prev",
    "estado_comercial",
]

top_cuentas_cali = (
    resumen_clientes[cols_view]
    .head(30)
    .copy()
)

# =========================================
# mostrar valores en millones COP
# =========================================

top_cuentas_cali = top_cuentas_cali.copy()

cols_millones = [
    "ventas_12m",
    "ventas_6m",
    "utilidad_12m",
    "ticket_promedio_12m",
]

for col in cols_millones:
    top_cuentas_cali[col] = (
        top_cuentas_cali[col] / 1_000_000
    ).round(1)

# opcional: porcentaje más limpio
top_cuentas_cali["margen_bruto_pct_12m"] = (
    top_cuentas_cali["margen_bruto_pct_12m"] * 100
).round(1)

top_cuentas_cali["variacion_ventas_12m_vs_prev"] = (
    top_cuentas_cali["variacion_ventas_12m_vs_prev"] * 100
).round(1)

# =========================================
# excluir clientes con actividad económica:
# "COMERCIO EN GENERAL"
# =========================================

# excluir:
# - COMERCIO EN GENERAL
# - COMERCIO DE RODAMIENTOS Y AFINES

top_cuentas_cali = (
    top_cuentas_cali[
        ~top_cuentas_cali["actividad_economica"].isin([
            "COMERCIO EN GENERAL",
            "COMERCIO DE RODAMIENTOS Y AFINES",
            "COMERCIO Y MANTENIMIENTO DE AUTOMOTORES",
        ])
    ]
    .copy()
    .reset_index(drop=True)
)

top_cuentas_cali.head(30)

,rank_ventas_12m,razonsocial,actividad_economica,ventas_12m,ventas_6m,utilidad_12m,margen_bruto_pct_12m,facturas_12m,skus_12m,ticket_promedio_12m,ultima_compra,dias_sin_comprar,variacion_ventas_12m_vs_prev,estado_comercial
0,3,PGI COLOMBIA LTDA.,PRODUCTOS TEXTILES,745.2,4.8,737.8,99.0,16.0,29.0,46.6,2026-12-02,1,-12.6,Activa
1,5,IDEMIA COLOMBIA S.A.S.,ARTES Y LITOGRAFIA,404.6,95.7,404.4,100.0,8.0,8.0,50.6,2026-12-02,1,315.9,Activa
2,6,CARTONES AMERICA S.A.,FABRICACION PRODUCTOS CARTON Y PAPEL,393.8,69.3,326.4,82.9,90.0,139.0,4.4,2026-12-03,0,-71.2,Activa
3,7,PLASTICOS ESPECIALES S.A.S,PRODUCTOS PLASTICOS,389.5,21.6,386.6,99.3,3.0,4.0,129.8,2026-10-02,62,NaN,En Riesgo
4,8,NESTLE DE COLOMBIA S.A.,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,366.3,85.2,360.4,98.4,15.0,81.0,24.4,2026-12-03,0,-66.8,Activa
5,9,FAREVA VILLA RICA S.A.S,FABRICACION DE QUIMICOS Y FARMACEUTICOS,326.6,0.0,325.2,99.6,9.0,18.0,36.3,2026-02-23,283,105.0,Dormida
6,13,SERMOTOR INGENIERIA S.A.S,TALLERES,244.6,11.5,215.1,88.0,41.0,161.0,6.0,2026-12-02,1,-61.6,Activa
7,16,URREA URBANO ANDRES JULIAN,None,192.3,0.5,191.2,99.4,8.0,10.0,24.0,2026-09-02,92,1931.9,En Riesgo
8,18,CABLES DE ENERGIA Y DE TELECOMUNICACIONES S.A....,OTROS TIPOS DE MANUFACTURA,173.4,70.1,167.7,96.7,32.0,59.0,5.4,2026-10-02,62,-57.4,En Riesgo
9,20,SOLUCIONES EN INGENIERIA ELECTROMECANICA SIEM ...,TALLERES,164.9,5.1,164.1,99.5,18.0,34.0,9.2,2026-07-01,155,99.3,Dormida


In [41]:
df_cali.iloc[1000]

nit                                               860026759
razonsocial                           CARTONES AMERICA S.A.
prefijo                                                 SC2
numero                                              1004744
fecha                                   2024-09-19 00:00:00
sucursal                                                  1
idconvenio                                                 
ordencompra                                      4500225342
sku                                                MB 32SKF
nombreproducto                          ARANDELA DE RETENC.
prefijo_1                                             MB 32
sufijo                                                  SKF
cantidad                                                3.0
idunidad                                                Und
bodega                                                   50
precio                                          108842.1708
preciousd                               

In [52]:
ventas_cliente

nit
860026759    212.977879
800201914    141.306030
900611187    113.867748
890300406     99.399968
860002130     70.260580
                ...    
14944000       0.002000
900262123      0.000000
802021585      0.000000
800022623      0.000000
830007248     -0.129200
Name: ventas_mm, Length: 582, dtype: float64